## Supervised State Dependent Gaussian Model
### Method
- Each max tempreture data point is accompanied by a weather type 
- Diffrentiate each data point depending on its weather type , i.e create a separate array for each type that containng the data points with that type
- Then calculate the mean for each type and its variance, to define a normal distribution for each type
- Use the already developed probaility transition matrix to genarate the next possible state(weather type) 
- Then use said weather type to genarate the possible next temp point based on the current state it is in.
- This algorithm for forcasting the next temp is simpler as it does not require transformations

In [84]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('seattle.csv',)

# Cleaning the data 

# Removing empty cells as this is weather data
df = df.dropna()

# Cleaning  wrong date formats  
df['date'] = pd.to_datetime(df['date'],format= 'mixed')

# Using the same test train split as the HMM model
split = int(len(df)*0.8)
data = df[:split]
test = df[split:]

In [85]:
# Estimating the emission distribution for each state by building a genaral function for it.

def GetStateMean():
    States = data['weather'].unique()
    EmmisionMean = []

    for i in States:
        counter = 0
        StateTotal = 0
        for n in range(len(data)):
            if data['weather'][n] == i:
                counter += 1 
                StateTotal = data['temp_max'][n] + StateTotal
        EmmisionMean.append(StateTotal/counter)

    return EmmisionMean

# Emmision Std
def GetStateStd():
    EmmisionDev = []
    States = data['weather'].unique()
    
    for state in States:
        StateTemps = data.loc[data['weather'] == state, 'temp_max']
        EmmisionDev.append(StateTemps.std())
    return EmmisionDev

print(GetStateMean())
print(GetStateStd())

[np.float64(14.134782608695646), np.float64(13.372693032015055), np.float64(19.099191919191938), np.float64(5.573076923076924), np.float64(16.52)]
[np.float64(7.889633631581361), np.float64(4.916340495324771), np.float64(7.803017644855513), np.float64(3.1091552154638653), np.float64(7.056800605274935)]


## The contents of MarkovChainForWeatherType for ease 

In [86]:
# Trying to find the differnt possible states and define the statespace

types = set(data['weather'])
print(types)

{'rain', 'fog', 'sun', 'snow', 'drizzle'}


In [87]:
# Simulation based forcasting algorithm use the path provided from Markov chain

Weather = data['weather'] #<- The State Space is as follows {'snow', 'rain', 'drizzle', 'fog', 'sun'}
types = set(data['weather']) 

# Creating a func to make the intial transition  matrix with null values.
def CreateTransMatrix(df):
    dim = len(set(data['weather']))
    TransMatrix = np.zeros((dim,dim), dtype=float)
    return TransMatrix


# Building a function to get the probability oof each entry in the transtion prob. matrix 
def GetCellProb(df,State_1,State_2):
    # State 1 is the state x_t and State two is x_t+1 > Through the markov property
    # Plan: Go through the entire array(Temp) then if at i its state_1 and at i+1 its state_2 then add to a counter 
    Counter_trans = 0  # <- Number of desierd trans.
    Counter = 0 # <- Number of total trans.  
    for i in range(len(df)-1):
        if df[i] == State_1:
            if df[i+1] == State_2:
                Counter_trans += 1
                Counter += 1
            else:
                Counter += 1

    return (Counter_trans/Counter) if Counter > 0 else 0 #<- incase there are no trans.
 

def GetTransitionProb(array):
    
    types = sorted(set(array))
    Matrix = CreateTransMatrix(array)

    for i in range(len(types)):
        for n in range(len(types)):
            Matrix[i,n] = GetCellProb(array,types[i],types[n])
    return Matrix


# Creating a Function that given a state returns an initial distribution
def GenerateInitialDist(state):
    null = [0,0,1,0,0]
    for i in range(len(types)):
        if types[i] == state:
            null[i] = 1
        else:
            null[i] = 0
    return null

# Creating a function that uses the forcast probability vector 
def ReturnNextState(InitialDistributon,TransistionMatrix):

    ProbArray = InitialDistribution @ TransistionMatrix
    
    next_state = np.random.choice(types, p=ProbArray)

    return next_state


# Initial distributio , selected arbitarily 
InitialDistribution = np.array([0,0,1,0,0]) #<- The state at the start is drizzel

CurrentState = 2

types = ['snow', 'rain', 'drizzle', 'fog', 'sun']

TransistionMatrix = np.array(GetTransitionProb(Weather))

# print("InitialDistribution:", InitialDistribution)
# print("type:", type(InitialDistribution))
print(ReturnNextState(InitialDistribution,TransistionMatrix))
# print(GenerateInitialDist('sun'))

drizzle


In [102]:
# Forecast the weather type and temp N stages/steps 

def TypeSimul(N,IntialState):
   Path = [IntialState]  
   for i in range(N):
      Path.append(ReturnNextState(GenerateInitialDist(Path[i]),TransistionMatrix))
   return Path

# Forcast the temp using gaussian distribution given TypeSimul state path
def GetIndex(state):
  for i in range(len(types)):
    if types[i] == state:
      return i

Path = TypeSimul(5,'sun')

def TempSimul(path):
 Std = GetStateStd()
 Mean = GetStateMean()
 TempPath = []

 for n in path:
  i = GetIndex(n)
  TempPath.append(np.random.normal(Mean[i],Std[i]))
 return TempPath


print(Path)
print(TempSimul(Path))

['sun', np.str_('sun'), np.str_('drizzle'), np.str_('sun'), np.str_('drizzle'), np.str_('drizzle')]
[26.171689303499917, 12.458911458977386, 23.33965937572681, 33.2383253490357, 32.12357172325442, 20.923669317099343]
